# MMP13 osteoarthritis co-expression study – revision analyses

This notebook reproduces the analyses added during peer review (CIMB revision). It reads the processed matrices cached by `01_discovery_analysis.ipynb` and the validation series loaded as in `02_validation_analysis.ipynb`, and writes the supplementary tables and figure source data to `MyDrive/mmp13_oa/06_revision/`.

**How to use**
1. Run *Section 0* (installs packages and writes `revision_analyses.py`).
2. *Section 1* loads the discovery matrices with the same configuration and grouping rules as the discovery notebook, the locked signature and discovery table, and the validation series.
3. Run *Section 2*. The reproduction gates must print `OK` (pooled g = 0.69; 1,510 / 407 robust partners; r(meta_z) with the saved discovery table > 0.999).
4. Run the remaining sections in order; the last cell writes a summary of all key values.

| Section | Analysis | Output (manuscript numbering) |
|---|---|---|
| 3 | Seed genes, matched random genes, network centrality | Table S6, Figure S5 |
| 4 | Cell-type composition | Table S7 |
| 5 | Hypertrophy correlation by tissue | Figure S6 |
| 6 | Co-expression in control tissue | Table S8 |
| 7 | Age range of OA donors | text |
| 8 | Sex models; adolescent controls excluded | Tables S9, S11 |
| 9 | CAMERA, sample-label permutation; GSE169077 excluded | Tables S10, S11 |
| 10 | Spearman-specific variance | Table S11 |
| 11 | Single-cell cluster means and sensitivity analysis | Figure 8e, Figure S7, Table S12 |

Output file names inside the notebook use the numbering of an earlier draft (e.g. `FigureS7_source_hypertrophy_by_tissue.csv` is now Figure S6); `results/README.md` maps them to the published numbering. No systematic GEO database search was performed in this project (see the manuscript, Section 2.1).


## 0. Setup

In [ ]:
!pip -q install GEOparse gseapy scanpy leidenalg igraph harmonypy biopython statsmodels mygene openpyxl pyarrow

In [ ]:
%%writefile revision_analyses.py
"""Revision analyses for the MMP13 co-expression manuscript (CIMB, round 1).

Every function works on a common data interface:

    DATA[name] = {"expr": DataFrame genes x samples (log2, one row per gene symbol),
                  "meta": DataFrame indexed by sample with columns
                          group   : 'OA' | 'control' | 'preserved'
                          age     : float or NaN
                          sex     : 'F' | 'M' | NaN
                          joint   : 'knee' | 'hip' | NaN
                          patient : patient id (used to pair GSE117999/GSE98918)
                          platform: platform id (GSE114007 has two)
                          tissue  : 'cartilage' | 'meniscus' | 'synovium' | 'bone'}
    COHORT[name] = cohort label (GSE117999 and GSE98918 share one label)

Use the objects from the ORIGINAL analysis notebook if you still have them; the
reproduction gates (`check_reproduction`) tell you whether the inputs match the
published numbers before any new result is produced.
"""
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

SEED = 20261113
rng_global = np.random.default_rng(SEED)

HYPERTROPHY = ["RUNX2", "COL10A1", "IHH", "ALPL", "VEGFA", "SPP1", "PTH1R"]
CATABOLIC = ["MMP13", "MMP3", "MMP1", "ADAMTS4", "ADAMTS5", "MMP9"]
ANABOLIC = ["COL2A1", "ACAN", "SOX9", "PRG4", "COMP", "TIMP3"]

# Cell-type marker genes (Table S7). Markers that are locked MMP13 partners are
# dropped automatically by `composition_scores(..., exclude=locked_genes)`.
MARKERS = {
    "chondrocyte": ["COL2A1", "ACAN", "SOX9", "COMP", "CHAD", "CILP", "FMOD", "CNMD"],
    "fibroblast": ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "PDGFRA", "FBLN1", "MFAP5"],
    "lining_fibroblast": ["PRG4", "HAS1", "CD55", "CLIC5", "HTRA4"],
    "osteoblast": ["SP7", "BGLAP", "IBSP", "COL1A1", "DMP1", "PHEX", "SOST"],
    "osteoclast": ["ACP5", "CTSK", "CALCR", "OSCAR", "DCSTAMP", "MMP9"],
    "macrophage": ["CD68", "CD163", "CSF1R", "C1QA", "C1QB", "MARCO", "MS4A7"],
    "T_cell": ["CD3D", "CD3E", "CD2", "CD247", "TRAC"],
    "B_plasma": ["MS4A1", "CD79A", "MZB1", "JCHAIN", "IGHM"],
    "endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "EMCN"],
    "pericyte_SMC": ["ACTA2", "MYH11", "RGS5", "NOTCH3", "CNN1"],
    "adipocyte": ["ADIPOQ", "PLIN1", "LEP", "FABP4", "CIDEA"],
}
Y_GENES = ["RPS4Y1", "DDX3Y", "KDM5D", "EIF1AY", "UTY"]


# --------------------------------------------------------------------- utils
def zrow(df):
    """z-score each gene across samples (within dataset)."""
    m = df.mean(axis=1); s = df.std(axis=1, ddof=1).replace(0, np.nan)
    return df.sub(m, axis=0).div(s, axis=0)


def _rank_std(X):
    """X: ndarray genes x n -> per-row ranks standardised to mean 0, norm 1."""
    R = np.apply_along_axis(stats.rankdata, 1, X)
    R = R - R.mean(axis=1, keepdims=True)
    nrm = np.linalg.norm(R, axis=1, keepdims=True)
    nrm[nrm == 0] = np.nan
    return R / nrm


def _residualise(R, C):
    """Regress covariates C (n x k) out of each row of R (genes x n)."""
    if C is None or C.shape[1] == 0:
        return R
    C1 = np.column_stack([np.ones(C.shape[0]), C])
    beta, *_ = np.linalg.lstsq(C1, R.T, rcond=None)
    E = R - (C1 @ beta).T
    E = E - E.mean(axis=1, keepdims=True)
    nrm = np.linalg.norm(E, axis=1, keepdims=True); nrm[nrm == 0] = np.nan
    return E / nrm


def select_samples(meta, which="OA", min_oa=8):
    """Replicates the published rule: OA samples, or all samples if < min_oa OA."""
    if which == "OA":
        oa = meta.index[meta.group == "OA"]
        if len(oa) < min_oa:
            return list(meta.index[meta.group.isin(["OA", "control"])])
        return list(oa)
    if which == "control":
        return list(meta.index[meta.group == "control"])
    raise ValueError(which)


# --------------------------------------------------- co-expression meta core
def cohort_z(DATA, COHORT, seeds, which="OA", min_oa=8, covariates=None,
             seed_vectors=None):
    """Fisher z of Spearman(seed, gene) per cohort, exactly as module C of MMP13-OA discovery analysis:
    genes with zero variance in the selected samples are dropped per dataset, rho clipped
    to +/-0.999, z averaged within cohort, cohort n = mean sample count of its datasets."""
    names = list(seed_vectors.keys()) if seed_vectors is not None else list(seeds)
    per_cohort = {s: {} for s in names}
    n_cohort = {}
    for ds, d in DATA.items():
        meta = d["meta"]; expr = d["expr"]
        smp = select_samples(meta, which, min_oa)
        if len(smp) < 4:
            continue
        X = expr[smp]
        X = X[X.notna().all(axis=1) & (X.std(axis=1) > 0)]
        G = _rank_std(X.values)
        C = covariates(ds, meta.loc[smp]) if covariates else None
        if C is not None:
            G = _residualise(G, C)
        if seed_vectors is not None:
            S = np.vstack([seed_vectors[s][ds] for s in names])
            S = _rank_std(S)
            if C is not None: S = _residualise(S, C)
        else:
            idx = [X.index.get_loc(s) if s in X.index else None for s in names]
            S = np.vstack([G[i] if i is not None else np.full(G.shape[1], np.nan) for i in idx])
        Rm = np.clip(S @ G.T, -0.999, 0.999)                 # seeds x genes
        Z = np.arctanh(Rm)
        c = COHORT[ds]
        for k, s in enumerate(names):
            per_cohort[s].setdefault(c, []).append(pd.Series(Z[k], index=X.index))
        n_cohort.setdefault(c, []).append(len(smp))
    out = {}
    for s in names:
        cols = {c: pd.concat(lst, axis=1).mean(axis=1) for c, lst in per_cohort[s].items()}
        out[s] = pd.DataFrame(cols)
    n = pd.Series({c: float(np.mean(v)) for c, v in n_cohort.items()})
    return out, n


def meta_from_z(Z, n, var_factor=1.0, sign_frac=0.75, **_ignored):
    """Cohort-level weighted Stouffer as in MMP13-OA discovery analysis module C:
    meta_rho = tanh(sum(z w) / sum(w)), meta_z = sum(z w) / sqrt(sum(w)), w = n - 3;
    genes need >= max(2, ceil(K/2)) cohorts; robust = FDR < 0.05 and >= 75% sign agreement.
    var_factor = 1.06 applies the Fieller variance 1.06/(n - 3) (meta_z / sqrt(1.06))."""
    Z = Z[[c for c in Z.columns if c in n.index]]
    W = n[Z.columns] - 3
    av = Z.notna()
    num = (Z.fillna(0) * W).sum(axis=1)
    res = pd.DataFrame({"meta_rho": np.tanh(num / (av * W).sum(axis=1)),
                        "meta_z": num / np.sqrt((av * W).sum(axis=1)) / np.sqrt(var_factor),
                        "n_cohorts": av.sum(axis=1)})
    res["same_sign_frac"] = (np.sign(Z).eq(np.sign(res.meta_rho), axis=0) & av).sum(axis=1) / res.n_cohorts
    res["p"] = 2 * stats.norm.sf(res.meta_z.abs())
    res = res[res.n_cohorts >= max(2, int(np.ceil(Z.shape[1] / 2)))].copy()
    res["fdr"] = multipletests(res.p, method="fdr_bh")[1]
    res["robust"] = (res.fdr < 0.05) & (res.same_sign_frac >= sign_frac)
    return res


def coexpr(DATA, COHORT, seed="MMP13", **kw):
    var_factor = kw.pop("var_factor", 1.0)
    Zd, n = cohort_z(DATA, COHORT, [seed], **kw)
    res = meta_from_z(Zd[seed].drop(index=seed, errors="ignore"), n, var_factor=var_factor)
    return res, n


def summarise(res):
    pos = res[res.robust & (res.meta_z > 0)]; neg = res[res.robust & (res.meta_z < 0)]
    return {"n_tested": len(res), "n_pos": len(pos), "n_neg": len(neg)}


def top_partners(res, k=150):
    r = res[res.robust]
    pos = r[r.meta_z > 0].sort_values("meta_z", ascending=False).index[:k]
    neg = r[r.meta_z < 0].sort_values("meta_z").index[:k]
    return list(pos), list(neg)


def check_reproduction(res, published=None, expect=(1510, 407)):
    """Compare with the published counts and, if given, with the saved T3 table (MMP13-OA discovery analysis)."""
    sm = summarise(res)
    ok = (sm["n_pos"], sm["n_neg"]) == tuple(expect)
    msg = f"robust partners: {sm['n_pos']} positive / {sm['n_neg']} negative (published {expect[0]} / {expect[1]}); genes tested {sm['n_tested']}"
    if published is not None:
        common = res.index.intersection(published.index)
        r = np.corrcoef(res.loc[common, "meta_z"], published.loc[common, "meta_z"])[0, 1]
        msg += f"; r(meta_z) with saved T3 = {r:.4f} over {len(common)} genes"
        ok = ok and r > 0.999
    print("Reproduction gate:", msg, "->", "OK" if ok else "MISMATCH - do not use new numbers until fixed")
    return ok


# ------------------------------------------------ R1-1: seeds and centrality
def matched_random_genes(DATA, COHORT, target="MMP13", k=1000, exclude=(), seed=SEED):
    """Genes matched to `target` on mean and SD percentile, averaged over datasets."""
    pm, ps = [], []
    for ds, d in DATA.items():
        smp = select_samples(d["meta"], "OA")
        X = d["expr"][smp]
        pm.append(X.mean(axis=1).rank(pct=True)); ps.append(X.std(axis=1).rank(pct=True))
    PM = pd.concat(pm, axis=1); PS = pd.concat(ps, axis=1)
    full = PM.notna().all(axis=1)
    PM, PS = PM[full].mean(axis=1), PS[full].mean(axis=1)
    d = np.sqrt((PM - PM[target]) ** 2 + (PS - PS[target]) ** 2)
    d = d.drop(index=[target, *exclude], errors="ignore").sort_values()
    pool = list(d.index[: max(5 * k, k)])
    rng = np.random.default_rng(seed)
    return list(rng.choice(pool, size=k, replace=False))


def validation_corr(VAL, seed, center_platform=False, group="OA"):
    """Spearman(seed, gene) within one group (default OA) of each validation dataset."""
    out = {}
    for ds, d in VAL.items():
        meta = d["meta"]; smp = list(meta.index[meta.group == group])
        X = d["expr"][smp]
        X = X[X.notna().all(axis=1)]
        if center_platform and meta.loc[smp, "platform"].nunique() > 1:
            pl = meta.loc[smp, "platform"]
            X = X.T.groupby(pl.values).transform(lambda v: v - v.mean()).T
        if seed not in X.index:
            continue
        G = _rank_std(X.values); s = G[X.index.get_loc(seed)]
        out[ds] = pd.Series(G @ s, index=X.index).drop(index=seed)
    return out


def seed_comparison(DATA, COHORT, VAL, named=("MMP13", "OLFML2B", "THY1", "POSTN", "FAP", "MMP3"),
                    n_random=1000, k=150):
    """Neighbourhood size and replication for named seeds and matched random genes."""
    rand = matched_random_genes(DATA, COHORT, "MMP13", n_random, exclude=named)
    seeds = list(named) + rand
    Zd, n = cohort_z(DATA, COHORT, seeds)
    rows = []
    vstats = {}
    for ds, d in VAL.items():   # precompute validation ranks once
        meta = d["meta"]; smp = list(meta.index[meta.group == "OA"])
        X = d["expr"][smp]; X = X[X.notna().all(axis=1)]
        vstats[ds] = (X.index, _rank_std(X.values))
    for s in seeds:
        res = meta_from_z(Zd[s].drop(index=s, errors="ignore"), n)
        sm = summarise(res); pos, neg = top_partners(res, k)
        row = {"seed": s, "type": "named" if s in named else "random", **sm}
        for ds, (idx, G) in vstats.items():
            if s not in idx:
                row[f"{ds}_pos_frac"] = np.nan; row[f"{ds}_genome_r"] = np.nan; continue
            v = pd.Series(G @ G[idx.get_loc(s)], index=idx).drop(index=s)
            pp = [g for g in pos if g in v.index]; nn = [g for g in neg if g in v.index]
            row[f"{ds}_pos_frac"] = float((v[pp] > 0).mean()) if pp else np.nan
            row[f"{ds}_neg_frac"] = float((v[nn] < 0).mean()) if nn else np.nan
            common = res.index.intersection(v.index)
            row[f"{ds}_genome_r"] = stats.spearmanr(res.loc[common, "meta_rho"], v[common])[0]
        rows.append(row)
    T = pd.DataFrame(rows).set_index("seed")
    T["n_robust"] = T.n_pos + T.n_neg
    rnd = T[T.type == "random"]
    pct = {c: float((rnd[c] < T.loc["MMP13", c]).mean() * 100)
           for c in T.columns if c not in ("type",) and T[c].dtype.kind in "fi"}
    return T, pct


def network_centrality(DATA, COHORT, n_top=5000, beta=6, force=("MMP13",)):
    """WGCNA-style weighted degree and eigenvector centrality on meta-correlations."""
    var = []
    for ds, d in DATA.items():
        smp = select_samples(d["meta"], "OA")
        var.append(d["expr"][smp].var(axis=1).rank(pct=True))
    V = pd.concat(var, axis=1)
    V = V[V.notna().all(axis=1)].mean(axis=1).sort_values(ascending=False)
    genes = list(dict.fromkeys(list(V.index[:n_top]) + [g for g in force if g in V.index]))
    num = np.zeros((len(genes), len(genes))); den = 0.0
    cohort_mats = {}
    for ds, d in DATA.items():
        smp = select_samples(d["meta"], "OA")
        G = np.nan_to_num(_rank_std(d["expr"].loc[genes, smp].values))
        cohort_mats.setdefault(COHORT[ds], []).append((np.arctanh(np.clip(G @ G.T, -0.999, 0.999)), len(smp)))
    for c, lst in cohort_mats.items():
        Zc = np.mean([m for m, _ in lst], axis=0); nc = np.mean([nn for _, nn in lst])
        num += (nc - 3) * Zc; den += (nc - 3)
    Rmeta = np.tanh(num / den)
    A = np.abs(Rmeta) ** beta; np.fill_diagonal(A, 0)
    k = A.sum(axis=1)
    vals, vecs = np.linalg.eigh(A)
    ev = np.abs(vecs[:, -1])
    T = pd.DataFrame({"weighted_degree": k, "eigenvector": ev}, index=genes)
    T["degree_rank"] = T.weighted_degree.rank(ascending=False).astype(int)
    T["eigen_rank"] = T.eigenvector.rank(ascending=False).astype(int)
    return T.sort_values("degree_rank")


# ------------------------------------------------- R1-2: cell-type composition
def composition_scores(DATA, exclude=(), min_markers=2):
    out = {}
    for ds, d in DATA.items():
        Z = zrow(d["expr"])
        sc = {}
        for ct, mk in MARKERS.items():
            g = [m for m in mk if m in Z.index and m not in exclude]
            if len(g) >= min_markers:
                sc[ct] = Z.loc[g].mean(axis=0)
        out[ds] = pd.DataFrame(sc)
    return out


def composition_covariates(scores, n_pc=2):
    """Returns a covariate function for cohort_z: first n_pc PCs of composition scores."""
    def f(ds, meta_sel):
        S = scores[ds].loc[meta_sel.index].dropna(axis=1)
        S = (S - S.mean()) / S.std(ddof=1).replace(0, np.nan)
        S = S.dropna(axis=1)
        U, s, Vt = np.linalg.svd(S.values, full_matrices=False)
        k = min(n_pc, max(0, S.shape[0] - 5))
        return U[:, :k] * s[:k]
    return f


def mmp13_vs_composition(DATA, scores):
    rows = []
    for ds, d in DATA.items():
        smp = select_samples(d["meta"], "OA")
        if "MMP13" not in d["expr"].index: continue
        m = d["expr"].loc["MMP13", smp]
        for ct in scores[ds].columns:
            r, p = stats.spearmanr(m, scores[ds].loc[smp, ct])
            rows.append({"dataset": ds, "tissue": d["meta"].tissue.iloc[0], "cell_type": ct,
                         "rho": r, "p": p, "n": len(smp)})
    return pd.DataFrame(rows)


def compare_meta(res_a, res_b, locked_pos, locked_neg):
    common = res_a.index.intersection(res_b.index)
    r = stats.spearmanr(res_a.loc[common, "meta_rho"], res_b.loc[common, "meta_rho"])[0]
    keep_pos = sum((g in res_b.index) and res_b.at[g, "fdr"] < 0.05 and res_b.at[g, "meta_z"] > 0 for g in locked_pos)
    keep_neg = sum((g in res_b.index) and res_b.at[g, "fdr"] < 0.05 and res_b.at[g, "meta_z"] < 0 for g in locked_neg)
    return {"genome_r": r, "pos_kept": keep_pos, "pos_total": len(locked_pos),
            "neg_kept": keep_neg, "neg_total": len(locked_neg), **summarise(res_b)}


def nnls_deconvolution(bulk_log2, ref_linear, genes=None):
    """Proportions of single-cell reference clusters in bulk samples (cartilage)."""
    from scipy.optimize import nnls
    B = 2 ** bulk_log2
    g = [x for x in (genes or ref_linear.index) if x in B.index and x in ref_linear.index]
    Rf = ref_linear.loc[g].values; Rf = Rf / Rf.sum(axis=0, keepdims=True)
    P = {}
    for s in B.columns:
        y = B.loc[g, s].values; y = y / y.sum()
        w, _ = nnls(Rf, y); P[s] = w / w.sum() if w.sum() > 0 else w
    return pd.DataFrame(P, index=ref_linear.columns).T


# ---------------------------------------------------- R1-3: hypertrophy by tissue
def hypertrophy_by_tissue(DATA, genes=HYPERTROPHY):
    rows, pooled = [], []
    for ds, d in DATA.items():
        smp = list(d["meta"].index[d["meta"].group == "OA"])
        if len(smp) < 8: continue       # as in Figure 4b (n = 117)
        Z = zrow(d["expr"][smp])
        g = [x for x in genes if x in Z.index and x != "MMP13"]
        score = Z.loc[g].mean(axis=0); m = Z.loc["MMP13"]
        r, p = stats.spearmanr(m, score)
        t = d["meta"].tissue.iloc[0]
        rows.append({"dataset": ds, "tissue": t, "n": len(smp), "rho": r, "p": p})
        pooled.append(pd.DataFrame({"tissue": t, "mmp13": m, "score": score}))
    P = pd.concat(pooled)
    for t, sub in list(P.groupby("tissue")) + [("all", P)]:
        r, p = stats.spearmanr(sub.mmp13, sub.score)
        lo, hi = np.tanh(np.arctanh(r) + np.array([-1, 1]) * 1.96 * np.sqrt(1.06 / (len(sub) - 3)))
        rows.append({"dataset": "pooled", "tissue": t, "n": len(sub), "rho": r, "p": p,
                     "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows), P


# --------------------------------------------- R1-4: control-tissue co-expression
def oa_vs_control_difference(DATA, COHORT, seed="MMP13"):
    Zoa, noa = cohort_z(DATA, COHORT, [seed], which="OA", min_oa=0)
    Zc, nc = cohort_z(DATA, COHORT, [seed], which="control")
    A, B = Zoa[seed], Zc[seed]
    cohorts = [c for c in A.columns if c in B.columns]
    D = pd.DataFrame({c: (A[c] - B[c]) / np.sqrt(1 / (noa[c] - 3) + 1 / (nc[c] - 3)) for c in cohorts})
    w = pd.Series({c: 1 / (1 / (noa[c] - 3) + 1 / (nc[c] - 3)) for c in cohorts})
    meas = D.notna()
    Zcomb = (D.fillna(0) * meas.mul(np.sqrt(w), axis=1)).sum(axis=1) / np.sqrt(meas.mul(w, axis=1).sum(axis=1))
    p = 2 * stats.norm.sf(np.abs(Zcomb))
    keep = meas.sum(axis=1) >= np.ceil(0.5 * len(cohorts))
    out = pd.DataFrame({"z_diff": Zcomb[keep], "p": p[keep.values]})
    out["fdr"] = multipletests(out.p, method="fdr_bh")[1]
    return out


# ------------------------------------------------------------- R1-6: sex
def infer_sex(expr):
    Z = zrow(expr)
    x = Z.loc["XIST"] if "XIST" in Z.index else pd.Series(0, index=Z.columns)
    y = [g for g in Y_GENES if g in Z.index]
    ys = Z.loc[y].mean(axis=0) if y else pd.Series(0, index=Z.columns)
    s = ys - x
    return pd.Series(np.where(s > 0, "M", "F"), index=expr.columns), s


def case_control_models(DATA, covars=("age",), extra=()):
    import statsmodels.formula.api as smf
    rows = []
    for ds, d in DATA.items():
        meta = d["meta"]; smp = list(meta.index[meta.group.isin(["OA", "control"])])
        df = meta.loc[smp].copy(); df["y"] = d["expr"].loc["MMP13", smp].values
        df["oa"] = (df.group == "OA").astype(int)
        cv = [c for c in (*covars, *extra) if c in df and df[c].notna().all() and df[c].nunique() > 1]
        f = "y ~ oa" + "".join(f" + C({c})" if df[c].dtype == object else f" + {c}" for c in cv)
        fit = smf.ols(f, df).fit()
        rows.append({"dataset": ds, "formula": f, "beta_oa": fit.params["oa"], "p_oa": fit.pvalues["oa"],
                     "std_effect": fit.params["oa"] / np.sqrt(fit.scale), "n": len(df)})
    return pd.DataFrame(rows)


# ------------------------------------------------- case-control meta-analysis
def hedges_g(a, b):
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * np.var(a, ddof=1) + (n2 - 1) * np.var(b, ddof=1)) / (n1 + n2 - 2))
    d = (np.mean(a) - np.mean(b)) / sp
    J = 1 - 3 / (4 * (n1 + n2) - 9)
    g = J * d
    v = (n1 + n2) / (n1 * n2) + g ** 2 / (2 * (n1 + n2))
    return g, v


def dl_hksj(g, v):
    g, v = np.asarray(g), np.asarray(v); k = len(g)
    w = 1 / v; mu_f = np.sum(w * g) / np.sum(w)
    Q = np.sum(w * (g - mu_f) ** 2)
    tau2 = max(0, (Q - (k - 1)) / (np.sum(w) - np.sum(w ** 2) / np.sum(w)))
    ws = 1 / (v + tau2); mu = np.sum(ws * g) / np.sum(ws)
    se = np.sqrt(np.sum(ws * (g - mu) ** 2) / ((k - 1) * np.sum(ws)))
    t = stats.t.ppf(0.975, k - 1)
    p = 2 * stats.t.sf(abs(mu / se), k - 1)
    I2 = max(0, (Q - (k - 1)) / Q) if Q > 0 else 0
    return {"g": mu, "ci_low": mu - t * se, "ci_high": mu + t * se, "p": p, "tau2": tau2, "I2": I2, "k": k}


def case_control_meta(DATA, COHORT, exclude=(), sample_filter=None):
    per = []
    for ds, d in DATA.items():
        if ds in exclude: continue
        meta = d["meta"]
        if sample_filter is not None: meta = sample_filter(ds, meta)
        a = d["expr"].loc["MMP13", meta.index[meta.group == "OA"]].values
        b = d["expr"].loc["MMP13", meta.index[meta.group == "control"]].values
        g, v = hedges_g(a, b)
        per.append({"dataset": ds, "cohort": COHORT[ds], "g": g, "v": v, "n_oa": len(a), "n_ctrl": len(b)})
    P = pd.DataFrame(per)
    C = P.groupby("cohort").agg(g=("g", "mean"), v=("v", lambda x: np.mean(np.sqrt(x)) ** 2))  # r = 1
    return P, dl_hksj(C.g.values, C.v.values)


# -------------------------------------------- R1-7: correlation-aware gene sets
def inter_gene_correlation(DATA, genes, which="OA"):
    vals, wts = [], []
    for ds, d in DATA.items():
        smp = select_samples(d["meta"], which)
        g = [x for x in genes if x in d["expr"].index]
        if len(g) < 3: continue
        X = d["expr"].loc[g, smp].dropna()
        # remove the linear MMP13 effect so the estimate reflects residual correlation
        Xc = X.values - X.values.mean(1, keepdims=True)
        if "MMP13" in d["expr"].index:
            m = d["expr"].loc["MMP13", smp].values.astype(float); m = (m - m.mean()) / m.std()
            Xc = Xc - np.outer(Xc @ m / len(m), m)
        C = np.corrcoef(Xc); iu = np.triu_indices_from(C, 1)
        vals.append(np.nanmean(C[iu])); wts.append(len(smp))
    return float(np.average(vals, weights=wts)) if vals else 0.0


def camera_pr(stat, sets, igc=None, DATA=None):
    """Pre-ranked CAMERA (Wu & Smyth 2012, parametric version of limma::cameraPR)."""
    stat = stat.dropna(); G = len(stat); rows = []
    for name, genes in sets.items():
        idx = stat.index.intersection(genes); m = len(idx)
        if m < 10 or m > 500: continue
        rho = igc[name] if isinstance(igc, dict) else (inter_gene_correlation(DATA, idx) if DATA is not None else (igc or 0.01))
        vif = 1 + (m - 1) * rho
        mean_set = stat[idx].mean(); mean_rest = stat.drop(idx).mean()
        s2 = stat.var(ddof=1)
        t = (mean_set - mean_rest) / np.sqrt(s2 * (vif / m + 1 / (G - m)))
        p = 2 * stats.t.sf(abs(t), G - 2)
        rows.append({"set": name, "n_genes": m, "inter_gene_cor": rho, "direction": "Up" if t > 0 else "Down", "t": t, "p": p})
    T = pd.DataFrame(rows); T["fdr"] = multipletests(T.p, method="fdr_bh")[1]
    return T.sort_values("p")


def permutation_gene_sets(DATA, COHORT, sets, n_perm=1000, seed=SEED, batch=100):
    """Sample-label permutation: permute MMP13 within each cohort (patients kept aligned)."""
    rng = np.random.default_rng(seed)
    base, n = coexpr(DATA, COHORT, "MMP13")
    obs = {k: base.meta_z.reindex(v).dropna().mean() for k, v in sets.items()}
    null = {k: [] for k in sets}
    # MMP13 vectors on the selected samples
    vec, patients = {}, {}
    for ds, d in DATA.items():
        smp = select_samples(d["meta"], "OA")
        X = d["expr"][smp]; X = X[X.notna().all(axis=1)]
        vec[ds] = X.loc["MMP13"].values
        patients[ds] = d["meta"].loc[smp, "patient"].astype(str).values
    for b0 in range(0, n_perm, batch):
        sv = {}
        for b in range(b0, min(n_perm, b0 + batch)):
            perm_by_cohort = {}
            sv[f"perm{b}"] = {}
            for ds in DATA:
                c = COHORT[ds]
                if c not in perm_by_cohort:
                    pts = np.unique(patients[ds]); perm_by_cohort[c] = dict(zip(pts, rng.permutation(pts)))
                mp = perm_by_cohort[c]
                order = {p: i for i, p in enumerate(patients[ds])}
                src = [order.get(mp.get(p, p), i) for i, p in enumerate(patients[ds])]
                sv[f"perm{b}"][ds] = vec[ds][src]
        Zd, nn = cohort_z(DATA, COHORT, None, seed_vectors=sv)
        for s in sv:
            mz = meta_from_z(Zd[s], nn).meta_z
            for k, v in sets.items():
                null[k].append(mz.reindex(v).dropna().mean())
    rows = []
    for k in sets:
        nl = np.array(null[k]); o = obs[k]
        if np.isnan(o) or len(nl) == 0: continue
        p = (1 + np.sum(np.abs(nl - nl.mean()) >= abs(o - nl.mean()))) / (1 + len(nl))
        rows.append({"set": k, "mean_meta_z": o, "null_mean": nl.mean(), "null_sd": nl.std(), "p_perm": p})
    T = pd.DataFrame(rows); T["fdr"] = multipletests(T.p_perm, method="fdr_bh")[1]
    return T.sort_values("p_perm")


# ---------------------------------------------------------- single-cell
def sc_pipeline(adata, n_hvg=2000, flavor="seurat", n_pcs=50, n_neighbors=15, resolution=0.6,
                mito_max=None, doublets=False, harmony_key=None, seed=7):
    """Defaults reproduce MMP13-OA validation module V3: min_genes 200, min_cells 3, normalize_total 1e4 + log1p
    (only if counts), 2,000 HVGs (seurat flavour), scale (max 10), PCA (50), neighbours (15),
    UMAP, Leiden (resolution 0.6, scanpy default flavour), seed 7."""
    import scanpy as sc
    ad = adata.copy()
    if mito_max is not None:
        ad.var["mt"] = ad.var_names.str.upper().str.startswith("MT-")
        sc.pp.calculate_qc_metrics(ad, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)
        ad = ad[ad.obs.pct_counts_mt < mito_max].copy()
    X = ad.X.max()
    is_counts = float(X) > 50
    sc.pp.filter_cells(ad, min_genes=200); sc.pp.filter_genes(ad, min_cells=3)
    if doublets and is_counts:
        sc.pp.scrublet(ad, random_state=seed)
        ad = ad[~ad.obs.predicted_doublet].copy()
    if is_counts:
        sc.pp.normalize_total(ad, target_sum=1e4); sc.pp.log1p(ad)
    elif float(X) > 20:
        sc.pp.log1p(ad)
    ad.raw = ad
    sc.pp.highly_variable_genes(ad, n_top_genes=n_hvg, flavor=flavor)
    adh = ad[:, ad.var.highly_variable].copy(); sc.pp.scale(adh, max_value=10)
    sc.tl.pca(adh, n_comps=n_pcs, random_state=seed)
    rep = "X_pca"
    if harmony_key:
        import harmonypy
        ho = harmonypy.run_harmony(adh.obsm["X_pca"], adh.obs, [harmony_key], random_state=seed)
        Zc = np.asarray(ho.Z_corr)
        adh.obsm["X_pca_harmony"] = Zc if Zc.shape[0] == adh.n_obs else Zc.T
        rep = "X_pca_harmony"
    sc.pp.neighbors(adh, n_neighbors=n_neighbors, use_rep=rep, random_state=seed)
    sc.tl.umap(adh, random_state=seed); sc.tl.leiden(adh, resolution=resolution, random_state=seed)
    ad.obs["leiden"] = adh.obs["leiden"]; ad.obsm["X_umap"] = adh.obsm["X_umap"]
    ad.uns["params"] = dict(min_genes=200, min_cells=3, target_sum=1e4, n_hvg=n_hvg, flavor=flavor, n_pcs=n_pcs,
                            n_neighbors=n_neighbors, resolution=resolution, mito_max=mito_max, doublets=doublets,
                            harmony_key=harmony_key, seed=seed, input_was_counts=is_counts)
    return ad


def cluster_mmp13(ad, gene="MMP13", n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    x = ad.raw[:, gene].X
    x = np.asarray(x.todense()).ravel() if hasattr(x, "todense") else np.asarray(x).ravel()
    rows = []
    for c, idx in ad.obs.groupby("leiden").indices.items():
        v = x[idx]
        boots = [rng.choice(v, len(v)).mean() for _ in range(n_boot)]
        rows.append({"cluster": c, "n_cells": len(v), "mean": v.mean(), "ci_low": np.percentile(boots, 2.5),
                     "ci_high": np.percentile(boots, 97.5), "pct_detected": 100 * (v > 0).mean()})
    return pd.DataFrame(rows).sort_values("mean", ascending=False)


# --------------------------------------------------------- PRISMA GEO search



In [ ]:
import os, json, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
import importlib, revision_analyses as RA
importlib.reload(RA)
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/mmp13_oa"
OUT = os.path.join(BASE, "06_revision"); os.makedirs(OUT, exist_ok=True)
VALUES = {}          # collected numbers for the placeholders
def put(key, val):
    VALUES[key] = val; print(f"{key}: {val}")

## 1. Load the data exactly as in MMP13-OA discovery analysis and MMP13-OA validation
The next three cells are copied from your original notebooks (configuration, loaders and grouping rules unchanged). They read the cached matrices from Drive, so GEO is only contacted if a cache file is missing.

In [ ]:
# --- MMP13-OA discovery analysis: paths (tables of this run go to 06_revision, v5 tables are not touched)
import re, datetime, warnings; warnings.filterwarnings("ignore")
DIRS = {"cache": os.path.join(BASE, "00_geo_cache"), "proc": os.path.join(BASE, "01_processed_v2"),
        "tables": os.path.join(OUT, "v5_rerun_tables")}
for d_ in DIRS.values(): os.makedirs(d_, exist_ok=True)
# [step-1-config]
TARGET = "MMP13"

DATASETS = {
    # GSE57218: 33 OA patients each contribute a lesioned ("OA") and a macroscopically preserved sample;
    # "OA" = lesioned sample (one per patient) vs healthy; preserved samples are used in the paired analysis (Module P)
    "GSE57218":  dict(tissue="Cartilage",        gpl="GPL6947",  cohort="GSE57218",
                      paired=dict(pair_re=r"raak_(\d+)_", case_re=r"disease state: oa\b",
                                  ref_re=r"disease state: preserved"), use=True),
    "GSE117999": dict(tissue="Cartilage",        gpl="GPL20844", cohort="APM-TKA knee cohort",
                      ctrl=r"arthroscopic partial meniscectomy", use=True),
    "GSE169077": dict(tissue="Cartilage",        gpl=None,       cohort="GSE169077", use=True),
    "GSE51588":  dict(tissue="Subchondral bone", gpl="GPL13497", cohort="GSE51588", use=True),
    "GSE55235":  dict(tissue="Synovium",         gpl="GPL96",    cohort="GSE55235", use=True),
    "GSE55457":  dict(tissue="Synovium",         gpl="GPL96",    cohort="GSE55457", use=True),
    "GSE12021":  dict(tissue="Synovium",         gpl="GPL96",    cohort="GSE55457", use=False),  # duplicate of GSE55457
    "GSE98918":  dict(tissue="Meniscus",         gpl="GPL20844", cohort="APM-TKA knee cohort",
                      ctrl=r"arthroscopic partial meniscectomy", use=True),
}

EXCLUDE_RE = r"rheumat|\bra\b"
CTRL_RE = (r"healthy|non[- ]?oa\b|non[- ]?osteoarthrit|without oa|\bcontrol|post[- ]?mortem|"
           r"normal (donor|control|joint|knee|cartilage|synovi|subchondral|bone|menisc|tissue|individual)|"
           r"(^|[:;,|]\s*)normal\b|disease state: normal|joint trauma")
OA_RE = r"osteoarthrit|\boa\b"

CATABOLIC   = ["MMP13", "MMP3", "MMP1", "ADAMTS4", "ADAMTS5", "MMP9"]
ANABOLIC    = ["COL2A1", "ACAN", "SOX9", "PRG4", "COMP", "TIMP3"]
HYPERTROPHY = ["RUNX2", "COL10A1", "IHH", "ALPL", "VEGFA", "SPP1", "PTH1R"]

MIN_OA_FOR_COEXP  = 8
MIN_OA_FOR_ENDO   = 20     # tertile groups ≥ 7 samples each
TOP_PARTNERS      = 150
GSEA_PERMUTATIONS = 1000
N_BOOT            = 1000
SEED = 7
DUP_R_THRESHOLD = 0.98     # residual sample–sample correlation flagging the same array in two series
# [step-2-load]
import numpy as np, pandas as pd
import GEOparse

def _symbol_column(tbl):
    for c in ["Gene Symbol", "GENE_SYMBOL", "Symbol", "ILMN_Gene", "gene_symbol", "GENE", "Gene symbol",
              "SYMBOL", "gene_assignment"]:
        if c in tbl.columns and tbl[c].notna().mean() > 0.3:
            return c
    for c in tbl.columns:
        if "symbol" in c.lower():
            return c
    raise ValueError(f"No gene-symbol column in {list(tbl.columns)[:15]}")

def _clean_symbol(s, col):
    if not isinstance(s, str) or not s.strip():
        return np.nan
    if col == "gene_assignment":
        parts = s.split("//")
        return parts[1].strip() if len(parts) > 1 else np.nan
    return re.split(r"\s*///\s*|\s*//\s*|;|,", s)[0].strip() or np.nan

def _annotation(gsms):
    rows = {}
    for k, g in gsms.items():
        md = g.metadata
        r = {"title": " ".join(md.get("title", [])), "source": " ".join(md.get("source_name_ch1", []))}
        for ch in md.get("characteristics_ch1", []):
            if ":" in ch:
                key, val = ch.split(":", 1)
                r[f"ch_{key.strip().lower()}"] = val.strip()
        r["description"] = " ".join(md.get("description", [])[:1])
        r["text"] = " | ".join([r["title"], r["source"]] + [f"{c[3:]}: {v}" for c, v in r.items()
                                                              if c.startswith("ch_")] + [r["description"]]).lower()
        rows[k] = r
    return pd.DataFrame.from_dict(rows, orient="index")

def load_raw(acc, cfg):
    """Gene-level log2 matrix + sample annotation (cached in 01_processed_v2)."""
    pq = os.path.join(DIRS["proc"], f"{acc}_expr.parquet")
    an = os.path.join(DIRS["proc"], f"{acc}_annotation.csv")
    if os.path.exists(pq) and os.path.exists(an):
        return pd.read_parquet(pq), pd.read_csv(an, index_col=0)
    gse = GEOparse.get_GEO(geo=acc, destdir=DIRS["cache"], silent=True)
    gpl_id = cfg.get("gpl") or list(gse.gpls)[0]
    gsms = {k: v for k, v in gse.gsms.items() if v.metadata.get("platform_id", [gpl_id])[0] == gpl_id}
    annot = _annotation(gsms)
    expr = pd.concat({k: g.table.set_index("ID_REF")["VALUE"] for k, g in gsms.items()}, axis=1)
    expr = expr.apply(pd.to_numeric, errors="coerce")
    ann = gse.gpls[gpl_id].table
    col = _symbol_column(ann)
    sym = ann.set_index("ID")[col].map(lambda s: _clean_symbol(s, col))
    expr.index = expr.index.astype(str); sym.index = sym.index.astype(str)
    expr = expr.join(sym.rename("SYMBOL"), how="inner").dropna(subset=["SYMBOL"])
    vals = expr.drop(columns="SYMBOL")
    if np.nanpercentile(vals.values, 99) > 50:          # not yet log-scale
        vals = np.log2(vals.clip(lower=0) + 1)
    vals["SYMBOL"] = expr["SYMBOL"]
    vals["_m"] = vals.drop(columns="SYMBOL").mean(axis=1)
    vals = vals.sort_values("_m", ascending=False).drop_duplicates("SYMBOL").drop(columns="_m")
    expr = vals.set_index("SYMBOL").dropna(how="all")
    expr = expr.loc[expr.notna().mean(axis=1) > 0.8]
    expr = expr.apply(lambda r: r.fillna(r.mean()), axis=1)
    expr.to_parquet(pq); annot.to_csv(an)
    return expr, annot

def assign_groups(annot, cfg):
    ctrl_re, oa_re = cfg.get("ctrl", CTRL_RE), cfg.get("oa", OA_RE)
    def f(t):
        if re.search(EXCLUDE_RE, t): return None
        if re.search(ctrl_re, t):   return "Control"
        if re.search(oa_re, t):     return "OA"
        return None
    grp = annot["text"].map(f)
    note = ""
    if cfg.get("oa_keep"):
        keep = annot["text"].str.contains(cfg["oa_keep"], regex=True)
        n_match = int((keep & (grp == "OA")).sum())
        if n_match >= 3:
            dropped = int(((grp == "OA") & ~keep).sum())
            grp[(grp == "OA") & ~keep] = None
            note = f"OA restricted to '{cfg['oa_keep']}' ({n_match} kept, {dropped} removed)"
        else:
            note = f"⚠️ oa_keep '{cfg['oa_keep']}' matched {n_match} OA samples → all OA samples kept"
            cols = [c for c in annot.columns if c not in ("text",)]
            print(f"   annotation values for this dataset (to set 'oa_keep' / 'patient_col'):")
            for c in cols:
                u = annot.loc[grp == "OA", c].astype(str).unique()
                print(f"     {c}: {len(u)} unique — {', '.join(u[:6])}{' …' if len(u) > 6 else ''}")
    return grp, note

DATA, AUDIT_ROWS, PAIRED_RAW = {}, [], {}
for acc, cfg in DATASETS.items():
    if not cfg.get("use", True):
        AUDIT_ROWS.append(dict(dataset=acc, tissue=cfg["tissue"], status="excluded in config"))
        continue
    try:
        expr, annot = load_raw(acc, cfg)
        grp, note = assign_groups(annot, cfg)
        if cfg.get("paired"):
            PAIRED_RAW[acc] = (expr.copy(), annot.copy(), cfg["paired"])
        grp = grp.dropna()
        expr = expr.loc[:, grp.index]
        n_c, n_o = int((grp == "Control").sum()), int((grp == "OA").sum())
        ok = TARGET in expr.index and n_c >= 3 and n_o >= 3
        AUDIT_ROWS.append(dict(dataset=acc, tissue=cfg["tissue"], cohort=cfg["cohort"], n_control=n_c, n_OA=n_o,
                               n_unassigned=int(len(annot) - len(grp)), genes=expr.shape[0], note=note,
                               status="used" if ok else "skipped"))
        pass
        if ok:
            DATA[acc] = dict(expr=expr, groups=grp, tissue=cfg["tissue"], cohort=cfg["cohort"],
                             gpl=cfg.get("gpl"))
        print(f"{acc:10s} {cfg['tissue']:17s} control={n_c:3d} OA={n_o:3d} unassigned={len(annot)-len(grp):3d} {note}")
        for g_ in ["Control", "OA"]:
            ex = annot.loc[grp[grp == g_].index[:1], "text"]
            if len(ex):
                print(f"{'':12s}e.g. {g_:7s}: {ex.iloc[0][:110]}")
    except Exception as e:
        AUDIT_ROWS.append(dict(dataset=acc, tissue=cfg["tissue"], status=f"error: {type(e).__name__}: {str(e)[:80]}"))
        print(f"{acc:10s} ERROR {e}")

# --- automatic duplicate-array check across series on the same platform
def duplicate_check(data, thr=DUP_R_THRESHOLD):
    found = []
    accs = list(data)
    for i, a in enumerate(accs):
        for b in accs[i + 1:]:
            if data[a]["gpl"] is None or data[a]["gpl"] != data[b]["gpl"]:
                continue
            common = data[a]["expr"].index.intersection(data[b]["expr"].index)
            X = pd.concat([data[a]["expr"].loc[common], data[b]["expr"].loc[common]], axis=1)
            X = X.sub(X.mean(axis=1), axis=0)
            Xa, Xb = X.iloc[:, :data[a]["expr"].shape[1]], X.iloc[:, data[a]["expr"].shape[1]:]
            Xa = (Xa - Xa.mean()) / Xa.std(); Xb = (Xb - Xb.mean()) / Xb.std()
            R = Xa.T.values @ Xb.values / (len(common) - 1)
            for jb in np.where(R.max(axis=0) > thr)[0]:
                ja = R[:, jb].argmax()
                found.append(dict(dataset_kept=a, sample_kept=Xa.columns[ja],
                                  dataset_dropped=b, sample_dropped=Xb.columns[jb], r=round(R[ja, jb], 4)))
    return pd.DataFrame(found)

DUPS = duplicate_check(DATA)
if len(DUPS):
    print(f"\n⚠️ {len(DUPS)} duplicated arrays across series → removed from the later series:")
    print(DUPS.to_string(index=False))
    for acc, s in DUPS.groupby("dataset_dropped"):
        keep = [c for c in DATA[acc]["expr"].columns if c not in set(s.sample_dropped)]
        DATA[acc]["expr"] = DATA[acc]["expr"][keep]; DATA[acc]["groups"] = DATA[acc]["groups"][keep]
        if min((DATA[acc]["groups"] == g).sum() for g in ["OA", "Control"]) < 3:
            print(f"   {acc} has <3 samples left in a group → dropped"); DATA.pop(acc)
else:
    print("\nDuplicate-array check: no duplicated arrays between series.")
DUPS.to_csv(os.path.join(DIRS["tables"], "S1_duplicate_array_check.csv"), index=False)

AUDIT = pd.DataFrame(AUDIT_ROWS)
for r in AUDIT.itertuples():
    if r.dataset in DATA:
        AUDIT.loc[r.Index, ["n_control", "n_OA"]] = [(DATA[r.dataset]["groups"] == "Control").sum(),
                                                    (DATA[r.dataset]["groups"] == "OA").sum()]
AUDIT.to_csv(os.path.join(DIRS["tables"], "S0_dataset_audit.csv"), index=False)
COHORT = pd.Series({a: d["cohort"] for a, d in DATA.items()})
print(f"\n{len(DATA)} datasets / {COHORT.nunique()} independent cohorts used.")
AUDIT
def get_age(acc):
    an = pd.read_csv(os.path.join(DIRS["proc"], f"{acc}_annotation.csv"), index_col=0)
    col = next((c for c in an.columns if re.search(r"^ch_(.*[^a-z])?age\b", c)), None)
    if col is None: return None
    return pd.to_numeric(an[col].astype(str).str.extract(r"(\d+\.?\d*)")[0], errors="coerce")


DATA_V5, COHORT_V5 = DATA, COHORT          # original objects


In [ ]:
# --- convert to the interface of revision_analyses.py
TISSUE_MAP = {"Cartilage": "cartilage", "Meniscus": "meniscus", "Synovium": "synovium", "Subchondral bone": "bone"}
DATA, COHORT, SUMMARY = {}, {}, []
for acc, d in DATA_V5.items():
    an = pd.read_csv(os.path.join(DIRS["proc"], f"{acc}_annotation.csv"), index_col=0).reindex(d["groups"].index)
    age = get_age(acc)
    age = age.reindex(d["groups"].index) if age is not None else pd.Series(np.nan, index=d["groups"].index)
    sexcol = next((c for c in an.columns if re.search(r"^ch_(sex|gender)\b", str(c))), None)
    sex = (an[sexcol].astype(str).str.strip().str.lower().str[0].map({"f": "F", "m": "M", "w": "F"})
           if sexcol else pd.Series(np.nan, index=an.index))
    joint = an["text"].astype(str).str.extract(r"\b(knee|hip)\b")[0]
    pcol = next((c for c in an.columns if re.search(r"^ch_(patient|donor|individual|subject)", str(c))), None)
    patient = an[pcol].astype(str) if pcol else pd.Series(an.index, index=an.index)
    meta = pd.DataFrame({"group": d["groups"].map({"OA": "OA", "Control": "control"}), "age": age, "sex": sex,
                         "joint": joint, "patient": patient, "platform": d.get("gpl") or "", 
                         "tissue": TISSUE_MAP[d["tissue"]]})
    DATA[acc] = {"expr": d["expr"][meta.index], "meta": meta}
    COHORT[acc] = d["cohort"]
    SUMMARY.append(dict(dataset=acc, tissue=d["tissue"], cohort=d["cohort"], OA=int((meta.group == "OA").sum()),
                        control=int((meta.group == "control").sum()), age_annotated=int(meta.age.notna().sum()),
                        sex_column=sexcol, sex_annotated=int(meta.sex.notna().sum()),
                        joint=dict(meta.joint.value_counts()), patient_column=pcol))
print(pd.DataFrame(SUMMARY).to_string(index=False))

# locked signature and saved discovery table
T3 = pd.read_csv(os.path.join(BASE, "02_tables_v5", "T3_MMP13_coexpression_consensus.csv"), index_col=0)
D = T3
LK = json.load(open(os.path.join(BASE, "05_validation", "tables", "V0_locked_signature.json")))
LOCKED_POS, LOCKED_NEG = LK["pos"], LK["neg"]
print(f"Locked signature ({LK['locked']}): {len(LOCKED_POS)} positive / {len(LOCKED_NEG)} negative")


In [ ]:
# --- MMP13-OA validation: validation series with the original loaders
import io, gzip, tarfile, zipfile, shutil, urllib.request
VDIR = {"cache": os.path.join(BASE, "05_validation", "cache")}
GEO_CACHE = os.path.join(BASE, "00_geo_cache")
from scipy import stats
# [step-1-config]
TARGET = "MMP13"
N_TOP = 150           # locked partner set size (same as discovery enrichment)
N_MODULE = 50         # module score size (same as discovery paired analysis)
MIN_OA = 8

BULK = {
    # accession: tissue, kind ('matrix' = series-matrix values; 'supp' = supplementary expression table), expected design
    "GSE114007": dict(tissue="Cartilage", kind="supp", expect="18 normal vs 20 OA (Fisch et al., 2018)", n=dict(OA=20, Control=18), use=True),
    "GSE82107":  dict(tissue="Synovium",  kind="matrix", expect="OA vs normal synovium", n=dict(OA=10, Control=7), use=True),
    "GSE89408":  dict(tissue="Synovium",  kind="supp", expect="not used in the manuscript", use=False),
}
SC = dict(acc="GSE104782", expect="OA knee cartilage chondrocytes (Ji et al., 2019)", use=True)

EXCLUDE_RE = r"rheumat|\bra\b|arthralgia|undifferentiated|early arthritis|\bua\b"
CTRL_RE = (r"healthy|non[- ]?oa\b|non[- ]?osteoarthrit|\bcontrol|post[- ]?mortem|normal|donor|"
           r"arthroscopic partial meniscectomy")
OA_RE = r"osteoarthrit|(?<![a-z0-9])oa(?![a-z0-9])"   # "oa_cart_1" counts; "goal" does not
GRADE_RE = r"\b(oa|kl|kellgren[- ]?lawrence|outerbridge|mankin|icrs)[ _-]*(grade|score|stage)\s*[:=]?\s*\S*"
HYPERTROPHY = ["RUNX2", "COL10A1", "IHH", "ALPL", "VEGFA", "SPP1", "PTH1R"]
SEED = 7
# [step-3-loaders]
import GEOparse

def _symbol_column(tbl):
    for c in ["Gene Symbol", "GENE_SYMBOL", "Symbol", "ILMN_Gene", "gene_symbol", "GENE", "Gene symbol", "SYMBOL",
              "gene_assignment"]:
        if c in tbl.columns and tbl[c].notna().mean() > 0.3:
            return c
    for c in tbl.columns:
        if "symbol" in c.lower():
            return c
    raise ValueError("no gene-symbol column")

def _clean(s, col):
    if not isinstance(s, str) or not s.strip(): return np.nan
    if col == "gene_assignment":
        p = s.split("//"); return p[1].strip() if len(p) > 1 else np.nan
    return re.split(r"\s*///\s*|\s*//\s*|;|,", s)[0].strip() or np.nan

def sample_table(gse):
    rows = {}
    for k, g in gse.gsms.items():
        md = g.metadata
        txt = " | ".join(md.get("title", []) + md.get("source_name_ch1", []) + md.get("characteristics_ch1", [])).lower()
        age = None
        for ch in md.get("characteristics_ch1", []):
            m = re.match(r"\s*age[^:]*:\s*([\d.]+)", ch.lower())
            if m: age = float(m.group(1))
        aliases = [d_.strip() for d_ in md.get("description", []) if d_.strip()] + \
                  [ch.split(":", 1)[1].strip() for ch in md.get("characteristics_ch1", []) if ":" in ch]
        rows[k] = dict(title=md.get("title", [""])[0], text=txt, age=age, aliases=aliases)
    return pd.DataFrame.from_dict(rows, orient="index")

def classify(t):
    t = re.sub(GRADE_RE, " ", t)      # "oa grade: 1" is a severity field, not a diagnosis
    if re.search(EXCLUDE_RE, t): return None
    if re.search(r"non[- ]?oa\b|non[- ]?osteoarthrit|without oa", t): return "Control"
    if re.search(OA_RE, t): return "OA"
    if re.search(CTRL_RE, t): return "Control"
    return None

def collapse(expr, sym):
    expr = expr.join(sym.rename("SYMBOL"), how="inner").dropna(subset=["SYMBOL"])
    v = expr.drop(columns="SYMBOL"); v["SYMBOL"] = expr["SYMBOL"]
    v["_m"] = v.drop(columns="SYMBOL").mean(axis=1)
    return v.sort_values("_m", ascending=False).drop_duplicates("SYMBOL").drop(columns="_m").set_index("SYMBOL")

def to_log(expr):
    x = expr.astype(float)
    if np.nanpercentile(x.values, 99) > 50:
        integer_like = np.nanmean(np.isclose(x.values[~np.isnan(x.values)][:200000] % 1, 0)) > 0.95
        if integer_like:                                  # raw counts → log2 CPM
            x = np.log2(x / x.sum(axis=0) * 1e6 + 1)
        else:
            x = np.log2(x.clip(lower=0) + 1)
    return x

def https(url):
    return url.replace("ftp://ftp.ncbi.nlm.nih.gov", "https://ftp.ncbi.nlm.nih.gov")

def fetch(url, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    fn = os.path.join(dest_dir, os.path.basename(url))
    if not os.path.exists(fn):
        urllib.request.urlretrieve(https(url), fn)
    return fn

def read_table_any(fn):
    low = fn.lower()
    if low.endswith((".xlsx", ".xls")):
        return pd.read_excel(fn)
    opener = gzip.open if low.endswith(".gz") else open
    with opener(fn, "rt", errors="replace") as fh:
        head = fh.read(20000)
    sep = "\t" if head.count("\t") > head.count(",") else ","
    return pd.read_csv(fn, sep=sep, compression="infer", low_memory=False)

def map_symbols(index):
    idx = pd.Index(index.astype(str))
    if idx.str.match(r"^ENSG\d+").mean() > 0.5:
        import mygene
        ids = idx.str.replace(r"\..*$", "", regex=True)
        res = mygene.MyGeneInfo().querymany(list(ids.unique()), scopes="ensembl.gene", fields="symbol",
                                            species="human", as_dataframe=True, verbose=False)
        mp = res["symbol"].dropna()
        mp = mp[~mp.index.duplicated()]
        return pd.Series(ids.map(mp).values, index=index), "Ensembl→symbol via MyGene.info"
    return pd.Series(idx.values, index=index), "symbols as provided"

def load_matrix(acc):
    gse = GEOparse.get_GEO(geo=acc, destdir=GEO_CACHE, silent=True)
    st = sample_table(gse)
    gpl = list(gse.gpls)[0]
    gsms = {k: v for k, v in gse.gsms.items() if v.metadata.get("platform_id", [gpl])[0] == gpl}
    expr = pd.concat({k: g.table.set_index("ID_REF")["VALUE"] for k, g in gsms.items()}, axis=1)
    expr = expr.apply(pd.to_numeric, errors="coerce")
    ann = gse.gpls[gpl].table; col = _symbol_column(ann)
    sym = ann.set_index("ID")[col].map(lambda s: _clean(s, col))
    expr.index = expr.index.astype(str); sym.index = sym.index.astype(str)
    expr = collapse(expr, sym)
    expr = to_log(expr).dropna(how="any")
    return expr, st.loc[expr.columns], gse.metadata.get("title", [""])[0], f"series matrix ({gpl})"

def load_supp(acc):
    gse = GEOparse.get_GEO(geo=acc, destdir=GEO_CACHE, silent=True)
    st = sample_table(gse)
    files = [f for f in gse.metadata.get("supplementary_file", []) if not f.lower().endswith((".tar", "filelist.txt"))]
    print(f"   supplementary files: {[os.path.basename(f) for f in gse.metadata.get('supplementary_file', [])]}")
    if not files:
        raise ValueError("no series-level supplementary expression table")
    parts, notes = [], []
    keys = {}
    for k_, r_ in st.iterrows():
        for v_ in [k_, r_["title"]] + r_.get("aliases", []):
            if isinstance(v_, str) and v_.strip(): keys.setdefault(v_.lower().strip(), set()).add(k_)
    for f in files:
        fn = fetch(f, os.path.join(VDIR["cache"], acc))
        try:
            t = read_table_any(fn)
        except Exception as e:
            print(f"   could not read {os.path.basename(fn)}: {e}"); continue
        gene_col = next((c for c in t.columns if not pd.api.types.is_numeric_dtype(t[c])), None)
        if gene_col is None: continue
        t = t.set_index(gene_col); num = t.select_dtypes(include=[np.number])
        if num.shape[1] < 3: continue
        cols = pd.Index(num.columns.astype(str)); mapped = {}
        for c in cols:
            cl = c.lower().strip()
            hit = keys.get(cl, set())
            if not hit:
                hit = {g for kk, gs in keys.items() for g in gs
                       if re.search(rf"(^|[_\s.\-|]){re.escape(cl)}($|[_\s.\-|])", kk)}
            if len(hit) == 1: mapped[c] = next(iter(hit))
        print(f"   {os.path.basename(fn)}: {len(mapped)}/{len(cols)} columns matched to GEO samples "
              f"(columns e.g. {list(cols[:4])}; titles e.g. {list(st['title'][:4])})")
        if len(mapped) >= 0.5 * len(cols):
            parts.append(num[list(mapped)].rename(columns=mapped)); notes.append(os.path.basename(fn))
    if not parts:
        raise ValueError("no supplementary table whose columns match the GEO samples — see printout")
    best = pd.concat([p.groupby(level=0).sum() if p.index.duplicated().any() else p for p in parts], axis=1, join="inner")
    best = best.loc[:, ~best.columns.duplicated()]
    note = " + ".join(notes)
    best = best.groupby(level=0).sum() if best.index.duplicated().any() else best
    sym, how = map_symbols(best.index)
    best.index = best.index.astype(str); sym.index = sym.index.astype(str)
    expr = collapse(best, sym)
    expr = to_log(expr).dropna(how="any")
    return expr, st.loc[expr.columns], gse.metadata.get("title", [""])[0], f"{note}; {how}"
VAL = {}
for acc, cfg in BULK.items():
    if not cfg["use"]: continue
    expr, st, title, how = (load_matrix if cfg["kind"] == "matrix" else load_supp)(acc)
    st["group"] = st["text"].map(classify)
    got = {g_: int((st.loc[st.index.intersection(expr.columns), "group"] == g_).sum()) for g_ in cfg["n"]}
    assert got == cfg["n"], f"{acc}: group sizes {got} differ from {cfg['n']}"
    gse = GEOparse.get_GEO(geo=acc, destdir=GEO_CACHE, silent=True)
    plat = pd.Series({k: v.metadata.get("platform_id", [""])[0] for k, v in gse.gsms.items()})
    st = st.loc[expr.columns]
    meta = pd.DataFrame({"group": st.group.map({"OA": "OA", "Control": "control"}), "age": st.age,
                         "sex": np.nan, "joint": np.nan, "patient": st.index, "platform": plat.reindex(st.index).values,
                         "tissue": cfg["tissue"].lower()}, index=st.index)
    meta = meta[meta.group.notna()]
    VAL[acc] = {"expr": expr[meta.index], "meta": meta}
    print(acc, how, meta.group.value_counts().to_dict(), "platforms:", meta.platform.value_counts().to_dict())


## 2. Reproduction gates (must pass before using any new number)

In [ ]:
P_cc, pooled = RA.case_control_meta(DATA, COHORT)
print(P_cc.round(3)); print({k: round(float(v), 3) for k, v in pooled.items()})
print("published: pooled g = 0.69 (95% CI -0.39 to 1.77), P = 0.16, I2 = 67%")

RES, N_COH = RA.coexpr(DATA, COHORT, "MMP13")
GATE = RA.check_reproduction(RES, published=T3)


## 3. R1-1 – Is MMP13 more central than other genes?

In [ ]:
T_seed, pct = RA.seed_comparison(DATA, COHORT, VAL, n_random=1000)
T_seed.to_csv(f"{OUT}/TableS6_seed_comparison.csv")
named = T_seed[T_seed.type == "named"]
print(named.round(3).to_string())
put("seed_MMP13_n_robust", int(T_seed.loc["MMP13", "n_robust"]))
for s in ["OLFML2B", "THY1", "POSTN", "FAP", "MMP3"]:
    put(f"seed_{s}_n_robust", int(T_seed.loc[s, "n_robust"]))
rnd = T_seed[T_seed.type == "random"]
put("random_median_n_robust", float(rnd.n_robust.median()))
put("MMP13_percentile_n_robust", round(float((rnd.n_robust < T_seed.loc['MMP13', 'n_robust']).mean() * 100), 1))
for c in [c for c in T_seed.columns if c.endswith("_pos_frac")]:
    put(f"MMP13_percentile_{c}", round(float((rnd[c] < T_seed.loc['MMP13', c]).mean() * 100), 1))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(rnd.n_robust, bins=40, color="grey"); ax[0].axvline(T_seed.loc["MMP13", "n_robust"], color="red")
for s in ["OLFML2B", "THY1", "POSTN", "FAP", "MMP3"]:
    ax[0].axvline(T_seed.loc[s, "n_robust"], ls="--", lw=0.8); ax[0].text(T_seed.loc[s, "n_robust"], ax[0].get_ylim()[1]*0.9, s, rotation=90, fontsize=7)
ax[0].set_xlabel("robust partners"); ax[0].set_title("MMP13 (red) vs 1,000 matched random genes")
NET = RA.network_centrality(DATA, COHORT, n_top=5000, beta=6)
NET.to_csv(f"{OUT}/TableS6b_network_centrality.csv")
ax[1].scatter(NET.weighted_degree, NET.eigenvector, s=3, c="grey"); ax[1].scatter(*NET.loc["MMP13", ["weighted_degree", "eigenvector"]], c="red")
ax[1].set_xlabel("weighted degree"); ax[1].set_ylabel("eigenvector centrality")
plt.tight_layout(); plt.savefig(f"{OUT}/FigureS6_seed_centrality.png", dpi=300)
put("MMP13_degree_rank", f"{int(NET.loc['MMP13','degree_rank'])} of {len(NET)}")
put("MMP13_eigen_rank", f"{int(NET.loc['MMP13','eigen_rank'])} of {len(NET)}")

## 4. R1-2 – Cell-type composition

In [ ]:
SC = RA.composition_scores(DATA, exclude=set(LOCKED_POS) | set(LOCKED_NEG))
used = {ct: [m for m in mk if m not in set(LOCKED_POS) | set(LOCKED_NEG)] for ct, mk in RA.MARKERS.items()}
pd.Series({k: ", ".join(v) for k, v in used.items()}).to_csv(f"{OUT}/TableS7a_markers.csv")
CORR = RA.mmp13_vs_composition(DATA, SC); CORR.to_csv(f"{OUT}/TableS7b_mmp13_vs_composition.csv")
meta_ct = CORR.groupby("cell_type").apply(lambda d: np.tanh(np.average(np.arctanh(d.rho.clip(-.999, .999)), weights=d.n - 3)))
print(meta_ct.round(3))
put("rho_MMP13_fibroblast", round(float(meta_ct["fibroblast"]), 2)); put("rho_MMP13_osteoblast", round(float(meta_ct["osteoblast"]), 2))
for npc in (1, 2):
    RES_ADJ, _ = RA.coexpr(DATA, COHORT, "MMP13", covariates=RA.composition_covariates(SC, npc))
    cmp = RA.compare_meta(RES, RES_ADJ, LOCKED_POS, LOCKED_NEG); print(npc, cmp)
    RES_ADJ.to_csv(f"{OUT}/TableS7c_coexpr_composition_adjusted_{npc}PC.csv")
put("composition_adj_genome_r", round(float(cmp["genome_r"]), 3))
put("composition_adj_pos_kept", f"{cmp['pos_kept']}/{cmp['pos_total']}"); put("composition_adj_neg_kept", f"{cmp['neg_kept']}/{cmp['neg_total']}")
# Optional: NNLS deconvolution of cartilage against GSE104782 cluster means (run Section 11 first)
# REF = pd.DataFrame({c: np.expm1(AD.raw[AD.obs.leiden == c].X.mean(0)).A1 for c in AD.obs.leiden.cat.categories}, index=AD.raw.var_names)
# for ds in ["GSE57218", "GSE117999"]:
#     smp = DATA[ds]["meta"].index[DATA[ds]["meta"].group == "OA"]
#     PROP = RA.nnls_deconvolution(DATA[ds]["expr"][smp], REF)
#     print(ds, PROP.corrwith(DATA[ds]["expr"].loc["MMP13", smp], method="spearman").round(2))

## 5. R1-3 – Hypertrophy score by tissue

In [ ]:
H, HP = RA.hypertrophy_by_tissue(DATA); print(H.round(3).to_string())
H2, _ = RA.hypertrophy_by_tissue(DATA, genes=[g for g in RA.HYPERTROPHY if g not in ("SPP1", "ALPL")])
H.to_csv(f"{OUT}/FigureS7_source_hypertrophy_by_tissue.csv"); H2.to_csv(f"{OUT}/FigureS7_source_noSPP1_ALPL.csv")
pool = H[H.dataset == "pooled"].set_index("tissue"); pool2 = H2[H2.dataset == "pooled"].set_index("tissue")
for t in pool.index: put(f"hyp_rho_{t}", f"{pool.loc[t,'rho']:.2f} (95% CI {pool.loc[t,'ci_low']:.2f} to {pool.loc[t,'ci_high']:.2f}; n = {int(pool.loc[t,'n'])})")
put("hyp_rho_all_noSPP1ALPL", round(float(pool2.loc["all", "rho"]), 2)); put("hyp_rho_cartilage_noSPP1ALPL", round(float(pool2.loc["cartilage", "rho"]), 2))
fig, axs = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
for a, t in zip(axs, ["cartilage", "meniscus", "synovium", "bone"]):
    d = HP[HP.tissue == t]; a.scatter(d.score, d.mmp13, s=10); a.set_title(f"{t} (rho = {pool.loc[t,'rho']:.2f})"); a.set_xlabel("hypertrophy score")
axs[0].set_ylabel("MMP13 (z)"); plt.tight_layout(); plt.savefig(f"{OUT}/FigureS7_hypertrophy_by_tissue.png", dpi=300)

## 6. R1-4 – Co-expression within control tissue

In [ ]:
RES_C, N_C = RA.coexpr(DATA, COHORT, "MMP13", which="control"); print(dict(N_C))
RES_C.to_csv(f"{OUT}/TableS8a_coexpr_control.csv")
common = RES.index.intersection(RES_C.index)
from scipy import stats
put("control_vs_OA_genome_r", round(float(stats.spearmanr(RES.loc[common, "meta_rho"], RES_C.loc[common, "meta_rho"])[0]), 3))
put("locked_pos_positive_in_controls_pct", round(float((RES_C.meta_rho.reindex(LOCKED_POS).dropna() > 0).mean() * 100), 1))
put("locked_neg_negative_in_controls_pct", round(float((RES_C.meta_rho.reindex(LOCKED_NEG).dropna() < 0).mean() * 100), 1))
DIFF = RA.oa_vs_control_difference(DATA, COHORT); DIFF.to_csv(f"{OUT}/TableS8b_OA_vs_control_difference.csv")
put("genes_differing_OA_vs_control_FDR05", int((DIFF.fdr < 0.05).sum()))
for ds, d in VAL.items():   # validation controls
    v = RA.validation_corr({ds: d}, "MMP13", group="control")[ds]
    put(f"{ds}_controls_locked_pos_positive_pct", round(float((v.reindex(LOCKED_POS).dropna() > 0).mean() * 100), 1))

## 7. R1-5 – Age distribution of OA donors

In [ ]:
rows = []
for ds, d in DATA.items():
    a = d["meta"].loc[d["meta"].group == "OA", "age"].dropna()
    if len(a): rows.append({"dataset": ds, "n": len(a), "median": a.median(), "q1": a.quantile(.25), "q3": a.quantile(.75), "min": a.min(), "max": a.max()})
AGE = pd.DataFrame(rows); print(AGE.to_string()); AGE.to_csv(f"{OUT}/age_OA_donors.csv")
put("OA_age_median_range", f"{AGE['median'].min():.0f}-{AGE['median'].max():.0f}"); put("OA_age_overall_range", f"{AGE['min'].min():.0f}-{AGE['max'].max():.0f}")

## 8. R1-6 sex and joint; R2 adolescent controls

In [ ]:
SEX = []
for ds, d in DATA.items():
    inferred, score = RA.infer_sex(d["expr"])
    ann = d["meta"].sex
    agree = (ann.dropna() == inferred[ann.dropna().index]).mean() if ann.notna().any() else np.nan
    SEX.append({"dataset": ds, "annotated_n": int(ann.notna().sum()), "agreement_with_inferred": agree})
    d["meta"]["sex_final"] = ann.fillna(inferred)
SEX = pd.DataFrame(SEX); print(SEX)
put("sex_annotated_datasets", int((SEX.annotated_n > 0).sum()))
M_age = RA.case_control_models(DATA, covars=("age",)); M_sex = RA.case_control_models(DATA, covars=("age", "sex_final"))
M = M_age.merge(M_sex, on="dataset", suffixes=("_age", "_age_sex")); print(M.round(4).to_string()); M.to_csv(f"{OUT}/TableS9_sex_models.csv")
# co-expression with age + sex as covariates
def cov_age_sex(ds, m):
    X = pd.DataFrame({"age": m.age, "sex": (m.sex_final == "M").astype(float)})
    X = X.loc[:, X.notna().all() & (X.nunique() > 1)]
    return X.values if X.shape[1] else None
RES_AS, _ = RA.coexpr(DATA, COHORT, "MMP13", covariates=cov_age_sex)
put("age_sex_adjusted_coexpr", RA.compare_meta(RES, RES_AS, LOCKED_POS, LOCKED_NEG))
# joint in GSE57218
if "GSE57218" in DATA:
    MJ = RA.case_control_models({"GSE57218": DATA["GSE57218"]}, covars=("age",), extra=("joint",)); print(MJ)
# adolescent controls removed
def drop_young(ds, m):
    return m[~((ds == "GSE57218") & (m.group == "control") & (m.age < 20))]
P_y, pooled_y = RA.case_control_meta(DATA, COHORT, sample_filter=drop_young)
g57 = P_y.set_index("dataset").loc["GSE57218"]
put("GSE57218_g_without_adolescent_controls", f"{g57.g:.2f} (95% CI {g57.g-1.96*np.sqrt(g57.v):.2f} to {g57.g+1.96*np.sqrt(g57.v):.2f}; controls n = {int(g57.n_ctrl)})")
put("pooled_g_without_adolescent_controls", {k: round(float(v), 3) for k, v in pooled_y.items()})

## 9. R1-7 – Correlation-aware gene-set tests and GSE169077 exclusion

In [ ]:
import gseapy as gp
HALL = gp.get_library("MSigDB_Hallmark_2020", organism="Human")
IGC = {k: RA.inter_gene_correlation(DATA, v) for k, v in HALL.items()}
CAM = RA.camera_pr(RES.meta_z, HALL, igc=IGC); CAM.to_csv(f"{OUT}/TableS10a_cameraPR.csv"); print(CAM.head(15).round(4).to_string())
PERM = RA.permutation_gene_sets(DATA, COHORT, HALL, n_perm=1000); PERM.to_csv(f"{OUT}/TableS10b_sample_permutation.csv"); print(PERM.head(15).round(4).to_string())
put("CAMERA_sets_FDR05", int((CAM.fdr < 0.05).sum())); put("permutation_sets_FDR05", int((PERM.fdr < 0.05).sum()))
put("CAMERA_FDR05_list", ", ".join(CAM[CAM.fdr < 0.05].set.head(12)))
# R check (optional): limma::cameraPR(statistic, index, inter.gene.cor = IGC[set])
RES.meta_z.to_csv(f"{OUT}/meta_z_for_R_cameraPR.csv"); pd.Series(IGC).to_csv(f"{OUT}/inter_gene_cor.csv")

DATA_no169 = {k: v for k, v in DATA.items() if k != "GSE169077"}
RES_no169, _ = RA.coexpr(DATA_no169, COHORT, "MMP13"); RES_no169.to_csv(f"{OUT}/TableS11a_coexpr_without_GSE169077.csv")
c = RA.compare_meta(RES, RES_no169, LOCKED_POS, LOCKED_NEG); print(c)
put("no169_pos_neg", f"{c['n_pos']} / {c['n_neg']}"); put("no169_genome_r", round(float(c["genome_r"]), 3))
robust_all = set(RES.index[RES.robust]); kept = len(robust_all & set(RES_no169.index[RES_no169.robust]))
put("no169_robust_partners_kept", f"{kept} of {len(robust_all)}")
_, pooled_no169 = RA.case_control_meta(DATA, COHORT, exclude=("GSE169077",))
put("pooled_g_without_GSE169077", f"{pooled_no169['g']:.2f} (95% CI {pooled_no169['ci_low']:.2f} to {pooled_no169['ci_high']:.2f}; P = {pooled_no169['p']:.2g}; I2 = {100*pooled_no169['I2']:.0f}%)")

## 10. R1 minor 5 – Spearman-specific variance 1.06/(n − 3)

In [ ]:
RES_106, _ = RA.coexpr(DATA, COHORT, "MMP13", var_factor=1.06); RES_106.to_csv(f"{OUT}/TableS11b_coexpr_fieller_variance.csv")
s = RA.summarise(RES_106); put("fieller_pos_neg", f"{s['n_pos']} / {s['n_neg']}")

## 11. Single-cell (GSE104782): parameters, cluster MMP13 (Figure 8e), sensitivity (Figure S8)
The loader is copied from MMP13-OA validation; `PRIMARY` holds the parameters of the submitted analysis (read from that notebook). The first run must reproduce 1,597 cells and 9 clusters; small differences can arise from newer Scanpy/leiden versions and are printed.

In [ ]:
SC = dict(acc="GSE104782", use=True)
# [module-V3]
import scanpy as sc, anndata
SCDIR = os.path.join(VDIR["cache"], SC["acc"])
def find_sc_inputs(acc):
    gse = GEOparse.get_GEO(geo=acc, destdir=GEO_CACHE, silent=True)
    print("title:", gse.metadata.get("title", [""])[0])
    sup = gse.metadata.get("supplementary_file", [])
    print("supplementary files:", [os.path.basename(f) for f in sup])
    paths = []
    for f in sup:
        if f.lower().endswith("filelist.txt"): continue
        fn = fetch(f, SCDIR); paths.append(fn)
        if fn.lower().endswith(".tar"):
            with tarfile.open(fn) as tf: tf.extractall(os.path.join(SCDIR, "untar"))
    return gse, paths

def load_sc(paths):
    root = os.path.join(SCDIR, "untar")
    h5 = [p for p in paths if p.endswith(".h5")] + ([os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs if f.endswith(".h5")] if os.path.isdir(root) else [])
    if h5:
        ads = [sc.read_10x_h5(p) for p in h5]
        for a, p in zip(ads, h5): a.obs["sample"] = os.path.basename(p).split("_")[0]; a.var_names_make_unique()
        return (anndata.concat(ads, join='outer', label='batch', index_unique='-') if len(ads) > 1 else ads[0]), f"10x h5 ({len(h5)} files)"
    mtx = [os.path.join(dp, f) for dp, _, fs in os.walk(SCDIR) for f in fs if re.search(r"matrix\.mtx(\.gz)?$", f)]
    if mtx:
        ads = []
        for m in mtx:
            d = os.path.dirname(m); pre = os.path.basename(m).replace("matrix.mtx.gz", "").replace("matrix.mtx", "")
            a = sc.read_10x_mtx(d, prefix=pre or None, var_names="gene_symbols"); a.obs["sample"] = pre or os.path.basename(d)
            a.var_names_make_unique(); ads.append(a)
        return (anndata.concat(ads, join='outer', label='batch', index_unique='-') if len(ads) > 1 else ads[0]), f"10x mtx ({len(mtx)} samples)"
    tabs = [p for p in paths if re.search(r"\.(txt|csv|tsv)(\.gz)?$", p.lower())] + \
           ([os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs if re.search(r"\.(txt|csv|tsv)(\.gz)?$", f.lower())]
            if os.path.isdir(root) else [])
    ads = []
    for p in tabs:
        t = read_table_any(p)
        gcol = next((c for c in t.columns if not pd.api.types.is_numeric_dtype(t[c])), None)
        if gcol is None: continue
        t = t.set_index(gcol).select_dtypes(include=[np.number])
        if t.shape[0] < 1000 or t.shape[1] < 50: continue
        if pd.Index(t.index.astype(str)).isin(D.index).mean() > 0.2:      # genes × cells
            a = sc.AnnData(t.T.values.astype(np.float32), obs=pd.DataFrame(index=t.columns.astype(str)),
                           var=pd.DataFrame(index=t.index.astype(str)))
        else:                                                              # cells × genes
            a = sc.AnnData(t.values.astype(np.float32), obs=pd.DataFrame(index=t.index.astype(str)),
                           var=pd.DataFrame(index=t.columns.astype(str)))
        a.var_names_make_unique(); a.obs["sample"] = os.path.basename(p).split(".")[0]; ads.append(a)
    if ads:
        return (anndata.concat(ads, join='outer', label='batch', index_unique='-') if len(ads) > 1 else ads[0]), f"dense tables ({len(ads)})"
    raise ValueError("no readable single-cell matrix (h5, mtx or dense table) found")


gse_sc, paths = find_sc_inputs(SC["acc"])
ADATA_RAW, how = load_sc(paths)
DONOR_KEY = "sample" if "sample" in ADATA_RAW.obs else "batch"
print("loaded", ADATA_RAW.shape, "from", how, "| donor/sample key:", DONOR_KEY, ADATA_RAW.obs[DONOR_KEY].nunique(), "levels")


In [ ]:
import scanpy as sc
PRIMARY = dict(n_hvg=2000, flavor="seurat", n_pcs=50, n_neighbors=15, resolution=0.6, seed=7)   # = MMP13-OA validation V3
AD = RA.sc_pipeline(ADATA_RAW, **PRIMARY)
print("cells", AD.n_obs, "clusters", AD.obs.leiden.nunique(), "(published: 1,597 cells, 9 clusters)")
CM = RA.cluster_mmp13(AD); CM.to_csv(f"{OUT}/Figure8e_source_cluster_MMP13.csv"); print(CM.round(3))
sc.tl.rank_genes_groups(AD, "leiden", method="wilcoxon", use_raw=True)
top_c = CM.iloc[0].cluster
print("highest-MMP13 cluster", top_c, "markers:", list(pd.DataFrame(AD.uns["rank_genes_groups"]["names"])[top_c][:10]))
fig, ax = plt.subplots(figsize=(5, 3.5))
CM2 = CM.sort_values("cluster", key=lambda s: s.astype(int))
ax.bar(CM2.cluster, CM2["mean"], yerr=[CM2["mean"] - CM2.ci_low, CM2.ci_high - CM2["mean"]], color="grey", capsize=3)
ax.set_xlabel("Leiden cluster"); ax.set_ylabel("mean log-normalised MMP13"); plt.tight_layout(); plt.savefig(f"{OUT}/Figure8e_cluster_MMP13.png", dpi=300)

SENS = RA.sc_pipeline(ADATA_RAW, **PRIMARY, mito_max=20, doublets=True, harmony_key=DONOR_KEY)
from sklearn.metrics import adjusted_rand_score
common = AD.obs_names.intersection(SENS.obs_names)
ari = adjusted_rand_score(AD.obs.loc[common, "leiden"], SENS.obs.loc[common, "leiden"])
CMS = RA.cluster_mmp13(SENS); CMS.to_csv(f"{OUT}/TableS12_sensitivity_cluster_MMP13.csv")
top = CMS.iloc[0].cluster
sc.tl.rank_genes_groups(SENS, "leiden", method="wilcoxon", use_raw=True)
markers = pd.DataFrame(SENS.uns["rank_genes_groups"]["names"])[top].head(20).tolist()
put("sc_primary", f"{AD.n_obs} cells, {AD.obs.leiden.nunique()} clusters; top-MMP13 cluster mean {CM.iloc[0]['mean']:.2f} vs next {(CM.iloc[1]['mean'] if len(CM) > 1 else float('nan')):.2f}")
put("sc_sensitivity", f"{SENS.n_obs} cells, {SENS.obs.leiden.nunique()} clusters, ARI = {ari:.2f}; top-MMP13 cluster {top} markers: {', '.join(markers[:10])}")
put("sc_input_was_counts", AD.uns["params"]["input_was_counts"])
put("sc_params_primary", AD.uns["params"]); put("sc_params_sensitivity", SENS.uns["params"])
sc.pl.umap(SENS, color=["leiden", "MMP13"], show=False); plt.savefig(f"{OUT}/FigureS8_sc_sensitivity.png", dpi=300)

## 13. Collect all placeholder values

In [ ]:
with open(f"{OUT}/PLACEHOLDER_VALUES.md", "w") as f:
    for k, v in VALUES.items(): f.write(f"- **{k}**: {v}\n")
print(open(f"{OUT}/PLACEHOLDER_VALUES.md").read())
import shutil; shutil.make_archive("/content/revision_outputs", "zip", OUT)
try:
    from google.colab import files; files.download("/content/revision_outputs.zip")
except Exception:
    pass